In [1]:
import pandas as pd

In [2]:
results = pd.read_pickle('../sim/data/simulation_results_20260314_133534.pkl')

FileNotFoundError: [Errno 2] No such file or directory: '../sim/data/simulation_results_20260314_133534.pkl'

In [ ]:
results.keys()

In [ ]:
micro = pd.DataFrame(results['micro_logs'])
macro = pd.DataFrame(results['macro_logs'])
macro = macro.drop(macro[macro['entity_id'].isna()].index)
macro = macro.drop(macro[macro['macro_name'] == 'Lead Created'].index)

macro = macro.drop(macro[macro['macro_name'] == 'Base Opportunity Created'].index)

micro['entity_id'] = micro['entity_id'] + '_' + micro['run_id'].astype(str)

macro['entity_id'] = macro['entity_id'] + '_' + macro['run_id'].astype(str)

In [ ]:
micro = micro.drop(micro[micro['success'] == False].index)

In [ ]:
micro[micro['entity_id'] == '6d60595e-6391-4a36-9a61-16844ca58657_0']

In [ ]:
macro[macro['entity_id'] == '6d60595e-6391-4a36-9a61-16844ca58657_0']

In [ ]:
leads_df = pd.DataFrame()
for i in range(20):
    leads = (
        pd.DataFrame(results['runs'][f'{i}']['leads'])
        .T
        .reset_index()
        .rename(columns={'index': 'entity_id'})
    )
    leads['entity_id'] = leads['entity_id'].values + f'_{str(i)}'
    leads_df = pd.concat([leads_df, leads])
leads_df = leads_df.rename(columns = {'converted':'won'})

In [ ]:
opps_df = pd.DataFrame()
for i in range(20):
    opps = (
        pd.DataFrame(results['runs']['0']['opportunities'])
        .T
        .reset_index()
        .rename(columns={'index': 'entity_id'})
    )
    opps['entity_id'] = opps['entity_id'].values + f'_{str(i)}'
    opps_df = pd.concat([opps_df, opps])

In [ ]:
df = pd.concat([leads_df, opps_df])

In [ ]:
df[df['entity_id'] == '6d60595e-6391-4a36-9a61-16844ca58657_0']

In [ ]:
df.columns

In [ ]:
macro[macro['entity_id'] == '6d60595e-6391-4a36-9a61-16844ca58657_0']

In [ ]:
macro.columns

In [ ]:
micro[micro['entity_id'] == '6d60595e-6391-4a36-9a61-16844ca58657_0']

In [ ]:
micro.columns

In [ ]:
crash()

In [ ]:
import pandas as pd
import numpy as np

# --- 1. Forward-fill macro probabilities for each deal ---

# macro_df has: ['day', 'rep_id', 'entity_id', 'macro_name', 'advanced', 'old_stage', 'new_stage', 'probability', 'run_id']

# First, keep only what we need
macro_state = macro[['entity_id', 'day', 'probability']].copy()
macro_state = macro_state.sort_values(['entity_id', 'day'])

In [ ]:
# Create a full day range per deal (cross join)
unique_deals = macro_state[['entity_id']].drop_duplicates()
min_day = macro_state['day'].min()
max_day = macro_state['day'].max()

In [ ]:
# Use integer days as cross join
all_days = pd.DataFrame({'day': range(min_day, max_day+1)})
all_days['key'] = 1
unique_deals['key'] = 1

In [ ]:
# Cross join to get every day for every deal
deal_days = unique_deals.merge(all_days, on='key').drop('key', axis=1)

In [ ]:
# Merge probabilities
trajectory = deal_days.merge(macro_state, on=['entity_id','day'], how='left')

In [ ]:
# Forward-fill probabilities per deal
trajectory['probability'] = trajectory.groupby('entity_id')['probability'].ffill()

In [ ]:
# Optional: interpolate for smooth transitions
trajectory['probability'] = trajectory.groupby('entity_id')['probability'].apply(lambda x: x.interpolate(method='linear'))

In [ ]:
# --- 2. Compute deal age (days since first action for that deal) ---
first_day = trajectory.groupby('entity_id')['day'].min().rename('first_day')
trajectory = trajectory.merge(first_day, on='entity_id')
trajectory['deal_age'] = trajectory['day'] - trajectory['first_day']

In [ ]:
# --- 3. Merge final outcome (won/lost) from entity_df if needed ---
trajectory = trajectory.merge(df[['entity_id','won']], on='entity_id', how='left')

In [ ]:
# --- 4. Clean-up ---
trajectory = trajectory[['entity_id','deal_age','probability','won']].sort_values(['entity_id','deal_age']).reset_index(drop=True)

# trajectory now contains:
# entity_id | deal_age | probability | won
# ready for plotting with deal age on x-axis

In [ ]:
# Correct: use str.contains directly on the Series
trajectory = trajectory[trajectory['entity_id'].astype(str).str.contains('_14')]

In [ ]:
trajectory

In [ ]:
plt.plot(trajectory['deal_age'], trajectory['probability'])

In [ ]:
import pandas as pd
import numpy as np

# --- Keep only needed columns ---
macro_state = macro[['entity_id', 'day', 'probability', 'new_stage']].copy()
macro_state = macro_state.sort_values(['entity_id', 'day'])

In [ ]:
# --- Identify deal start and end ---
first_day = macro_state.groupby('entity_id')['day'].min()
# Optional: define closed stages (you may have stage names in macro_df)
closed_stages = ['Closed Won', 'Closed Lost']  # adjust to your stage names
# last day = last day the deal reaches a closed stage, or last day in macro_df if not closed yet
last_day = macro_state[macro_state['new_stage'].isin(closed_stages)].groupby('entity_id')['day'].max()
# For deals that never reach a closed stage, fill with last macro action day
last_day = last_day.combine_first(macro_state.groupby('entity_id')['day'].max())

In [ ]:
import pandas as pd
import numpy as np

# macro_df: ['entity_id','day','probability','new_stage']
trajectory_list = []

for deal_id in tqdm(macro['entity_id'].unique()):
    deal_data = macro[macro['entity_id'] == deal_id].sort_values('day').copy()
    
    # reset deal age = 0,1,2,... for each action
    deal_data['deal_age'] = (deal_data['day'] - deal_data['day'].min()).astype(int)
    
    # create full timeline from 0 → last action
    max_age = deal_data['deal_age'].max()
    full_days = pd.DataFrame({'deal_age': np.arange(max_age+1)})
    
    # merge probability
    deal_traj = full_days.merge(deal_data[['deal_age','probability']], on='deal_age', how='left')
    
    # forward fill and interpolate
    deal_traj['probability'] = deal_traj['probability'].ffill().interpolate()
    
    # add entity_id for merging
    deal_traj['entity_id'] = deal_id
    
    trajectory_list.append(deal_traj)

# combine all deals
trajectory = pd.concat(trajectory_list, ignore_index=True)

# optional: merge final outcome
trajectory = trajectory.merge(df[['entity_id','won']], on='entity_id', how='left')

In [ ]:
# Aggregate micro action impacts per deal per day
micro_impact = micro.groupby(['entity_id','day'])['success'].sum().reset_index()
# scale to small probability changes
micro_impact['delta_prob'] = micro_impact['success'] * 0.01  # adjust scaling factor

# Merge into trajectory
trajectory = trajectory.merge(micro_impact[['entity_id','day','delta_prob']], 
                              left_on=['entity_id','deal_age'], right_on=['entity_id','day'], 
                              how='left')
trajectory['delta_prob'] = trajectory['delta_prob'].fillna(0)
trajectory['probability'] = (trajectory['probability'] + trajectory['delta_prob']).clip(0,1)
trajectory.drop(['delta_prob','day_y'], axis=1, errors='ignore', inplace=True)

In [ ]:
trajectory

In [ ]:
trajectory.fillna(0)

In [ ]:
trajectory = trajectory[trajectory['entity_id'].astype(str).str.contains('_14')]

In [ ]:
plt.plot(trajectory['deal_age'], trajectory['probability'], linewidth = 0.25, alpha = 0.5)
# plt.xscale('symlog')